# RASCALS / V5 detector upgrade

V4 initial-warning filtering improved precision but reduced fresh-test early recall and was rejected for deployment. V5 preserves first static warnings and filters only persistent repeat notifications. Current breaches bypass the filter; validation selects the rising-trend guard. Fresh 500000-series final seeds are generated only after freeze. V4 outputs and source are retained as development history.

In [1]:
from pathlib import Path
import os,sys,platform,json
ROOT=Path.cwd()
if not (ROOT/'continuityguard').is_dir():ROOT=ROOT.parent
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
assert platform.machine()=='s390x'
print(platform.machine(),platform.python_version())
OUT=ROOT/'artifacts/v5-candidate'

s390x 3.12.15


## Fit and select using training/calibration/validation only

In [2]:
from continuityguard.upgrade import fit_select
selected=fit_select(OUT)

Generating training, calibration and validation only; final test not generated.


Fit and validate logistic_C0.1


{"candidate": "logistic_C0.1", "best_validation": {"forecast_threshold": 1.01, "warning_gate": 0.97, "feasible_vs_static": true, "objective": 0.7632321859903382}, "precision": 0.45893719806763283, "early_recall": 0.9270833333333334}


Fit and validate logistic_C1


{"candidate": "logistic_C1", "best_validation": {"forecast_threshold": 1.01, "warning_gate": 0.97, "feasible_vs_static": true, "objective": 0.7632321859903382}, "precision": 0.45893719806763283, "early_recall": 0.9270833333333334}


...

Fit and validate boosted_trees_15_leaves


{"candidate": "boosted_trees_15_leaves", "best_validation": {"forecast_threshold": 1.01, "warning_gate": 0.5, "feasible_vs_static": true, "objective": 0.7616950757575758}, "precision": 0.45454545454545453, "early_recall": 0.9270833333333334}


Fit and validate boosted_trees_31_leaves


{"candidate": "boosted_trees_31_leaves", "best_validation": {"forecast_threshold": 1.01, "warning_gate": 0.35, "feasible_vs_static": true, "objective": 0.7616950757575758}, "precision": 0.45454545454545453, "early_recall": 0.9270833333333334}


FROZEN before final evaluation: {"version": "cg-2026-10-17-v5", "trained_utc": "2026-10-17T13:07:45.438758+00:00", "architecture": "s390x", "model": "logistic_C0.1", "forecast_horizon_seconds": 60, "feature_count": 26, "sha256": "ca4328b8ed2099ea4a007e75e626a4a6db63344d5c492c344733ac6f9f4d65b5", "packages": {"numpy": "2.0.2", "scikit-learn": "1.9.0", "scipy": "1.13.1", "joblib": "1.6.0"}}


## Frozen fresh evaluation
The next cell verifies the previously written hashes before generating final runs. No policy adjustment follows final-test inspection.

In [3]:
from continuityguard.upgrade import evaluate_frozen
report=evaluate_frozen(OUT)
print('Frozen policy:',report['policy'])

Generating predeclared fresh final and stress runs AFTER freeze.


{
  "Static thresholds": {
    "incidents": 135,
    "alerts": 308,
    "detected": 134,
    "early_detections": 130,
    "late_detections": 4,
    "missed": 1,
    "incident_precision": 0.43506493506493504,
    "incident_recall": 0.9925925925925926,
    "early_warning_recall": 0.9629629629629629,
    "median_early_lead_seconds": 11.0,
    "healthy_service_hours": 30.0,
    "healthy_false_alerts": 0,
    "false_alarms_per_healthy_service_hour": 0.0,
    "unmatched_alerts_all_runs": 174
  },
  "Robust statistics": {
    "incidents": 135,
    "alerts": 1210,
    "detected": 135,
    "early_detections": 133,
    "late_detections": 2,
    "missed": 0,
    "incident_precision": 0.1115702479338843,
    "incident_recall": 1.0,
    "early_warning_recall": 0.9851851851851852,
    "median_early_lead_seconds": 26.0,
    "healthy_service_hours": 30.0,
    "healthy_false_alerts": 38,
    "false_alarms_per_healthy_service_hour": 1.2666666666666666,
    "unmatched_alerts_all_runs": 1075
  },
  "Isola

Paired uncertainty: {"unit": "whole run; paired detector differences; simulator-conditional", "replicates": 500, "early_recall_delta_95": [0.0, 0.0], "precision_delta_95": [0.006298946269637273, 0.03154657913211283]}


Frozen policy: {'mode': 'repeat_filtered', 'forecast_threshold': 1.01, 'standalone_forecast_paging': False, 'warning_gate': 0.97, 'trend_guard': False, 'novelty_threshold': 0.5967641153063866, 'novelty_quantile': 0.995, 'refresh_seconds': 60}
